# Original DINOv2 Food-101 Training — Stage Extract

The notebook copies original source cells **68, 69, 70, 74, 75, 76, 98, 99**, preserving the sequence from new classifier head through last-two and last-four transformer-block fine-tuning. Cell 74/76 ensemble evaluation needs a trained ConvNeXt `convnext_stage3_best.pth` checkpoint. The final last-four training also needs the preceding DINOv2 `dinov2_last2_best.pth` checkpoint and `fusion_improved_validation.pth`, as in the historical workflow. These are **original source extracts, not new completed experiments or guarantees of exact rerun scores**. GPU with bfloat16 support is required by the final stage. Run in order; downloading the public pretrained model is separate from the released tuned checkpoints.


In [ ]:
# Colab / Food-101 setup matching original experiment cell 4
%pip -q install -U timm tqdm
import os
from pathlib import Path
import torch
from torchvision.datasets import Food101
from torch.utils.data import random_split
from google.colab import drive

drive.mount('/content/drive')
save_dir = Path('/content/drive/MyDrive/NutriVision_AI')
save_dir.mkdir(parents=True, exist_ok=True)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
base = Food101(root='/content/data', split='train', download=True)
if len(base) != 75750:
    raise RuntimeError(f'Unexpected Food-101 training set size: {len(base)}')
split_file = save_dir / 'data_split.pth'
if split_file.exists():
    split = torch.load(split_file, map_location='cpu', weights_only=True)
else:
    val_size = int(len(base) * 0.10)
    train_ds, val_ds = random_split(base, [len(base)-val_size, val_size], generator=torch.Generator().manual_seed(42))
    split = {'train_indices': list(train_ds.indices), 'val_indices': list(val_ds.indices)}
    torch.save(split, split_file)
assert len(split['train_indices']) == 68175 and len(split['val_indices']) == 7575
assert not set(split['train_indices']).intersection(split['val_indices'])
print('Training:', len(split['train_indices']), 'Validation:', len(split['val_indices']), 'Device:', device)


## Initialize DINOv2 (source cell 68)

In [ ]:

import gc
import torch
import timm

# Free GPU memory from ConvNeXt
if "fusion_cnn" in globals():
    del fusion_cnn

if "outputs" in globals():
    del outputs

gc.collect()
torch.cuda.empty_cache()

# Reproducible classifier initialization
torch.manual_seed(42)

dino_name = "vit_base_patch14_dinov2.lvd142m"

# Load pretrained DINOv2 with a new Food-101 classifier
dino_model = timm.create_model(
    dino_name,
    pretrained=True,
    num_classes=101,
    img_size=280
).to(device)

dino_model.eval()

# Test a small batch before training
test_images = torch.randn(
    2, 3, 280, 280,
    device=device
)

with torch.inference_mode():
    with torch.autocast(
        device_type="cuda",
        enabled=(device.type == "cuda")
    ):
        test_outputs = dino_model(test_images)

print("Model:", dino_name)
print("Input:", test_images.shape)
print("Output:", test_outputs.shape)
print("Parameters:", round(
    sum(p.numel() for p in dino_model.parameters()) / 1e6, 2
), "million")

if device.type == "cuda":
    print(
        "GPU allocated:",
        round(torch.cuda.memory_allocated() / 1024**3, 2),
        "GB"
    )

assert test_outputs.shape == (2, 101)
print("DINOv2 ready!")


## Prepare Food-101 loaders (source cell 69)

In [ ]:

import torch
import timm
from torchvision import datasets, transforms as T
from torch.utils.data import DataLoader, Subset

# Use DINOv2 pretrained normalization
dino_cfg = timm.data.resolve_model_data_config(dino_model)

mean = dino_cfg["mean"]
std = dino_cfg["std"]

dino_train_transform = T.Compose([
    T.RandomResizedCrop(
        280,
        scale=(0.7, 1.0),
        interpolation=T.InterpolationMode.BICUBIC
    ),
    T.RandomHorizontalFlip(),
    T.ColorJitter(
        brightness=0.1,
        contrast=0.1,
        saturation=0.1
    ),
    T.ToTensor(),
    T.Normalize(mean, std)
])

dino_val_transform = T.Compose([
    T.Resize(
        320,
        interpolation=T.InterpolationMode.BICUBIC
    ),
    T.CenterCrop(280),
    T.ToTensor(),
    T.Normalize(mean, std)
])

dino_train_data = datasets.Food101(
    root="/content/data",
    split="train",
    transform=dino_train_transform,
    download=False
)

dino_val_data = datasets.Food101(
    root="/content/data",
    split="train",
    transform=dino_val_transform,
    download=False
)

# Reuse exactly the same train/validation split
dino_train = Subset(
    dino_train_data,
    split["train_indices"]
)

dino_val = Subset(
    dino_val_data,
    split["val_indices"]
)

dino_train_loader = DataLoader(
    dino_train,
    batch_size=16,
    shuffle=True,
    num_workers=0,
    pin_memory=True
)

dino_val_loader = DataLoader(
    dino_val,
    batch_size=16,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

print("Training images:", len(dino_train))
print("Validation images:", len(dino_val))
print("Number of classes:", len(dino_train_data.classes))
print("DINOv2 data ready!")


## Train new classifier head (source cell 70)

In [ ]:

import torch
import torch.nn as nn
from tqdm.auto import tqdm

# Freeze all pretrained layers
for param in dino_model.parameters():
    param.requires_grad = False

# Allow only the classifier to learn
for param in dino_model.head.parameters():
    param.requires_grad = True

criterion = nn.CrossEntropyLoss(
    label_smoothing=0.1
)

optimizer = torch.optim.AdamW(
    dino_model.head.parameters(),
    lr=5e-4,
    weight_decay=0.01
)

use_amp = device.type == "cuda"
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

epochs = 3
best_dino_acc = -1.0

for epoch in range(1, epochs + 1):

    # Frozen backbone remains in evaluation mode
    dino_model.eval()
    dino_model.head.train()

    train_correct = 0
    train_total = 0
    train_loss_sum = 0.0

    for images, labels in tqdm(
        dino_train_loader,
        desc=f"DINO Train {epoch}"
    ):

        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(
            device_type=device.type,
            enabled=use_amp
        ):
            outputs = dino_model(images)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        bs = labels.size(0)

        train_loss_sum += loss.item() * bs
        train_correct += (
            outputs.argmax(dim=1) == labels
        ).sum().item()
        train_total += bs

    # Validation
    dino_model.eval()

    val_correct = 0
    val_total = 0
    val_loss_sum = 0.0

    with torch.inference_mode():
        for images, labels in tqdm(
            dino_val_loader,
            desc=f"DINO Validation {epoch}"
        ):

            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            with torch.autocast(
                device_type=device.type,
                enabled=use_amp
            ):
                outputs = dino_model(images)
                loss = criterion(outputs, labels)

            bs = labels.size(0)

            val_loss_sum += loss.item() * bs
            val_correct += (
                outputs.argmax(dim=1) == labels
            ).sum().item()
            val_total += bs

    train_acc = 100 * train_correct / train_total
    val_acc = 100 * val_correct / val_total

    print(f"\n===== DINOv2 Epoch {epoch}/{epochs} =====")
    print(f"Train Loss: {train_loss_sum/train_total:.4f}")
    print(f"Train Accuracy: {train_acc:.2f}%")
    print(f"Validation Loss: {val_loss_sum/val_total:.4f}")
    print(f"Validation Accuracy: {val_acc:.2f}%")

    if val_acc > best_dino_acc:

        best_dino_acc = val_acc

        torch.save({
            "epoch": epoch,
            "model_state_dict": dino_model.state_dict(),
            "val_accuracy": val_acc,
            "classes": dino_train_data.classes,
            "model_name": dino_name,
            "image_size": 280
        }, f"{save_dir}/dinov2_head_best.pth")

        print("NEW BEST DINOv2 SAVED!")

    print(f"Best DINOv2 Accuracy: {best_dino_acc:.2f}%")

print("\nDINOv2 head training completed!")
print(f"Best Accuracy: {best_dino_acc:.2f}%")


## Build head-stage ConvNeXt–DINOv2 comparison (source cell 74)

In [ ]:

import os
import torch
import timm

from google.colab import drive
from torchvision import datasets, transforms as T
from torch.utils.data import DataLoader, Subset
from tqdm.auto import tqdm

# ======================================
# 1. RESTORE ENVIRONMENT
# ======================================

if not os.path.ismount("/content/drive"):
    drive.mount("/content/drive")

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

save_dir = "/content/drive/MyDrive/NutriVision_AI"

split = torch.load(
    f"{save_dir}/data_split.pth",
    map_location="cpu",
    weights_only=True
)

print("Device:", device)

# ======================================
# 2. LOAD CONVNEXT CHECKPOINT
# ======================================

conv_ckpt = torch.load(
    f"{save_dir}/convnext_stage3_best.pth",
    map_location="cpu",
    weights_only=True
)

conv_model = timm.create_model(
    "convnext_base.fb_in22k_ft_in1k_384",
    pretrained=False,
    num_classes=101
)

conv_model.load_state_dict(
    conv_ckpt["model_state_dict"]
)

conv_cfg = timm.data.resolve_model_data_config(
    conv_model
)

conv_transform = timm.data.create_transform(
    **conv_cfg,
    is_training=False
)

conv_model = conv_model.to(device).eval()

print("ConvNeXt loaded!")

# ======================================
# 3. LOAD DINOV2 CHECKPOINT
# ======================================

dino_ckpt = torch.load(
    f"{save_dir}/dinov2_head_best.pth",
    map_location="cpu",
    weights_only=True
)

dino_model = timm.create_model(
    "vit_base_patch14_dinov2.lvd142m",
    pretrained=False,
    num_classes=101,
    img_size=280
)

dino_model.load_state_dict(
    dino_ckpt["model_state_dict"]
)

dino_cfg = timm.data.resolve_model_data_config(
    dino_model
)

dino_transform = T.Compose([
    T.Resize(
        320,
        interpolation=T.InterpolationMode.BICUBIC
    ),
    T.CenterCrop(280),
    T.ToTensor(),
    T.Normalize(
        dino_cfg["mean"],
        dino_cfg["std"]
    )
])

dino_model = dino_model.to(device).eval()

print("DINOv2 loaded!")

# ======================================
# 4. RESTORE FOOD-101 DATASET
# ======================================

print("Checking Food-101 dataset...")

conv_data = datasets.Food101(
    root="/content/data",
    split="train",
    transform=conv_transform,
    download=True
)

dino_data = datasets.Food101(
    root="/content/data",
    split="train",
    transform=dino_transform,
    download=False
)

assert conv_data.classes == dino_data.classes
assert list(conv_ckpt["classes"]) == list(dino_ckpt["classes"])
assert list(conv_ckpt["classes"]) == list(conv_data.classes)

val_indices = split["val_indices"]

conv_val = Subset(conv_data, val_indices)
dino_val = Subset(dino_data, val_indices)

conv_loader = DataLoader(
    conv_val,
    batch_size=8,
    shuffle=False,
    num_workers=0
)

dino_loader = DataLoader(
    dino_val,
    batch_size=8,
    shuffle=False,
    num_workers=0
)

print("Validation images:", len(conv_val))
print("Dataset ready!")

# ======================================
# 5. RUN BOTH MODELS
# ======================================

conv_probs = []
dino_probs = []
all_labels = []

with torch.inference_mode():

    for (xc, yc), (xd, yd) in tqdm(
        zip(conv_loader, dino_loader),
        total=len(conv_loader),
        desc="NutriVision Fusion"
    ):

        assert torch.equal(yc, yd)

        xc = xc.to(device)
        xd = xd.to(device)

        with torch.autocast(
            device_type=device.type,
            enabled=(device.type == "cuda")
        ):
            out_conv = conv_model(xc)
            out_dino = dino_model(xd)

        conv_probs.append(
            torch.softmax(out_conv.float(), dim=1).cpu()
        )

        dino_probs.append(
            torch.softmax(out_dino.float(), dim=1).cpu()
        )

        all_labels.append(yc)

conv_probs = torch.cat(conv_probs)
dino_probs = torch.cat(dino_probs)
labels = torch.cat(all_labels)

# ======================================
# 6. COMPARE ACCURACY
# ======================================

def accuracy(probs):
    return 100 * (
        probs.argmax(1) == labels
    ).float().mean().item()

print("\n===== INDIVIDUAL MODELS =====")
print(f"ConvNeXt: {accuracy(conv_probs):.2f}%")
print(f"DINOv2: {accuracy(dino_probs):.2f}%")

print("\n===== ENSEMBLE RESULTS =====")

best_accuracy = -1
best_weight = None

for weight in [0.25, 0.50, 0.75]:

    fused = (
        weight * conv_probs
        + (1 - weight) * dino_probs
    )

    acc = accuracy(fused)

    print(
        f"ConvNeXt {weight:.0%} + "
        f"DINOv2 {1-weight:.0%}: {acc:.2f}%"
    )

    if acc > best_accuracy:
        best_accuracy = acc
        best_weight = weight

# ======================================
# 7. ANALYZE ERRORS
# ======================================

conv_correct = conv_probs.argmax(1) == labels
dino_correct = dino_probs.argmax(1) == labels

print("\n===== ERROR ANALYSIS =====")
print("Both correct:",
      (conv_correct & dino_correct).sum().item())

print("Only ConvNeXt correct:",
      (conv_correct & ~dino_correct).sum().item())

print("Only DINOv2 correct:",
      (~conv_correct & dino_correct).sum().item())

print("Both wrong:",
      (~conv_correct & ~dino_correct).sum().item())

print("\nBEST ENSEMBLE:", round(best_accuracy, 2), "%")
print("Best ConvNeXt weight:", best_weight)

# ======================================
# 8. SAVE PREDICTIONS
# ======================================

torch.save({
    "conv_probs": conv_probs,
    "dino_probs": dino_probs,
    "labels": labels,
    "classes": conv_data.classes,
    "best_weight": best_weight
}, f"{save_dir}/fusion_validation_predictions.pth")

print("\nResults saved successfully!")


## Fine-tune the last two transformer blocks (source cell 75)

In [ ]:

import gc
import torch
import torch.nn as nn

from torchvision import datasets, transforms as T
from torch.utils.data import DataLoader, Subset
from tqdm.auto import tqdm

# Free GPU memory occupied by ConvNeXt
if "conv_model" in globals():
    conv_model.cpu()
    del conv_model

gc.collect()
torch.cuda.empty_cache()

# Restore the best DINOv2 head checkpoint
head_ckpt = torch.load(
    f"{save_dir}/dinov2_head_best.pth",
    map_location="cpu",
    weights_only=True
)

dino_model.load_state_dict(head_ckpt["model_state_dict"])
dino_model = dino_model.to(device)

# Freeze pretrained parameters
for p in dino_model.parameters():
    p.requires_grad = False

# Unfreeze final two transformer blocks
for p in dino_model.blocks[-2:].parameters():
    p.requires_grad = True

# Unfreeze final normalization and classifier
for p in dino_model.norm.parameters():
    p.requires_grad = True

for p in dino_model.head.parameters():
    p.requires_grad = True

print("Trainable parameters:", sum(
    p.numel() for p in dino_model.parameters()
    if p.requires_grad
))

# Restore training augmentation
dino_cfg = timm.data.resolve_model_data_config(dino_model)

train_transform = T.Compose([
    T.RandomResizedCrop(
        280,
        scale=(0.7, 1.0),
        interpolation=T.InterpolationMode.BICUBIC
    ),
    T.RandomHorizontalFlip(),
    T.ColorJitter(
        brightness=0.1,
        contrast=0.1,
        saturation=0.1
    ),
    T.ToTensor(),
    T.Normalize(
        dino_cfg["mean"],
        dino_cfg["std"]
    )
])

train_data = datasets.Food101(
    root="/content/data",
    split="train",
    transform=train_transform,
    download=False
)

train_set = Subset(train_data, split["train_indices"])

train_loader = DataLoader(
    train_set,
    batch_size=8,
    shuffle=True,
    num_workers=0,
    pin_memory=(device.type == "cuda")
)

val_loader = DataLoader(
    dino_val,
    batch_size=8,
    shuffle=False,
    num_workers=0,
    pin_memory=(device.type == "cuda")
)

# Optimizer with differential learning rates
optimizer = torch.optim.AdamW([
    {
        "params": dino_model.blocks[-2:].parameters(),
        "lr": 1e-5
    },
    {
        "params": dino_model.norm.parameters(),
        "lr": 1e-5
    },
    {
        "params": dino_model.head.parameters(),
        "lr": 5e-5
    }
], weight_decay=0.01)

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

epochs = 3
accumulation_steps = 2

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=epochs,
    eta_min=1e-6
)

use_amp = device.type == "cuda"
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

best_accuracy = head_ckpt["val_accuracy"]

for epoch in range(1, epochs + 1):

    dino_model.train()
    optimizer.zero_grad(set_to_none=True)

    train_correct = 0
    train_total = 0
    train_loss_sum = 0

    for step, (images, labels) in enumerate(
        tqdm(train_loader, desc=f"Fine-tune {epoch}")
    ):

        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        with torch.autocast(
            device_type=device.type,
            enabled=use_amp
        ):
            outputs = dino_model(images)
            loss = criterion(outputs, labels)

        scaler.scale(
            loss / accumulation_steps
        ).backward()

        if (
            (step + 1) % accumulation_steps == 0
            or (step + 1) == len(train_loader)
        ):
            scaler.unscale_(optimizer)

            torch.nn.utils.clip_grad_norm_(
                filter(
                    lambda p: p.requires_grad,
                    dino_model.parameters()
                ),
                max_norm=1.0
            )

            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)

        bs = labels.size(0)
        train_correct += (outputs.argmax(1) == labels).sum().item()
        train_total += bs
        train_loss_sum += loss.item() * bs

    # Validation
    dino_model.eval()

    val_correct = 0
    val_total = 0
    val_loss_sum = 0

    with torch.inference_mode():
        for images, labels in tqdm(
            val_loader,
            desc=f"Validate {epoch}"
        ):
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            with torch.autocast(
                device_type=device.type,
                enabled=use_amp
            ):
                outputs = dino_model(images)
                loss = criterion(outputs, labels)

            bs = labels.size(0)
            val_correct += (outputs.argmax(1) == labels).sum().item()
            val_total += bs
            val_loss_sum += loss.item() * bs

    train_acc = 100 * train_correct / train_total
    val_acc = 100 * val_correct / val_total
    val_loss = val_loss_sum / val_total

    scheduler.step()

    print(f"\n===== DINOv2 Fine-tuning {epoch}/{epochs} =====")
    print(f"Train Accuracy: {train_acc:.2f}%")
    print(f"Validation Accuracy: {val_acc:.2f}%")
    print(f"Validation Loss: {val_loss:.4f}")

    if val_acc > best_accuracy:
        best_accuracy = val_acc

        torch.save({
            "epoch": epoch,
            "model_state_dict": dino_model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "val_accuracy": val_acc,
            "val_loss": val_loss,
            "classes": train_data.classes,
            "model_name": "vit_base_patch14_dinov2.lvd142m",
            "image_size": 280
        }, f"{save_dir}/dinov2_last2_best.pth")

        print("NEW BEST DINOv2 SAVED!")

    print(f"Best DINOv2 Accuracy: {best_accuracy:.2f}%")

print("\nFine-tuning completed!")
print(f"Best DINOv2 Accuracy: {best_accuracy:.2f}%")


## Evaluate and save improved validation ensemble (source cell 76)

In [ ]:

import os
import torch
import torch.nn.functional as F
import timm

from google.colab import drive
from torchvision import datasets, transforms as T
from torch.utils.data import DataLoader, Subset
from tqdm.auto import tqdm

# 1. Restore environment
if not os.path.ismount("/content/drive"):
    drive.mount("/content/drive")

save_dir = "/content/drive/MyDrive/NutriVision_AI"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 2. Load previously saved ConvNeXt predictions
old = torch.load(
    f"{save_dir}/fusion_validation_predictions.pth",
    map_location="cpu",
    weights_only=True
)

conv_probs = old["conv_probs"].float()
labels = old["labels"].long()

# 3. Load improved DINOv2 checkpoint
ckpt = torch.load(
    f"{save_dir}/dinov2_last2_best.pth",
    map_location="cpu",
    weights_only=True
)

dino_model = timm.create_model(
    "vit_base_patch14_dinov2.lvd142m",
    pretrained=False,
    num_classes=101,
    img_size=280
)

dino_model.load_state_dict(ckpt["model_state_dict"])
dino_model = dino_model.to(device).eval()

# 4. Same validation preprocessing used during training
cfg = timm.data.resolve_model_data_config(dino_model)

val_transform = T.Compose([
    T.Resize(320, interpolation=T.InterpolationMode.BICUBIC),
    T.CenterCrop(280),
    T.ToTensor(),
    T.Normalize(cfg["mean"], cfg["std"])
])

data = datasets.Food101(
    root="/content/data",
    split="train",
    transform=val_transform,
    download=True
)

split = torch.load(
    f"{save_dir}/data_split.pth",
    map_location="cpu",
    weights_only=True
)

assert list(old["classes"]) == list(ckpt["classes"])
assert list(old["classes"]) == list(data.classes)

val_set = Subset(data, split["val_indices"])

loader = DataLoader(
    val_set,
    batch_size=16,
    shuffle=False,
    num_workers=0,
    pin_memory=(device.type == "cuda")
)

# 5. Generate predictions from improved DINOv2
new_probs = []
new_labels = []

with torch.inference_mode():
    for images, y in tqdm(loader, desc="Improved DINOv2"):

        images = images.to(device)

        with torch.autocast(
            device_type=device.type,
            enabled=(device.type == "cuda")
        ):
            outputs = dino_model(images)

        new_probs.append(
            torch.softmax(outputs.float(), dim=1).cpu()
        )
        new_labels.append(y)

dino_probs = torch.cat(new_probs)
check_labels = torch.cat(new_labels)

# Critical: predictions must refer to identical images
assert torch.equal(labels, check_labels)
assert conv_probs.shape == dino_probs.shape

# 6. Evaluate accuracy and cross-entropy
def metrics(probs):
    accuracy = 100 * (
        probs.argmax(1) == labels
    ).float().mean().item()

    top5 = 100 * (
        probs.topk(5, dim=1).indices == labels[:, None]
    ).any(1).float().mean().item()

    ce = F.nll_loss(
        probs.clamp_min(1e-12).log(),
        labels
    ).item()

    return accuracy, top5, ce

print("\n===== MODEL COMPARISON =====")

for name, probs in [
    ("ConvNeXt", conv_probs),
    ("Improved DINOv2", dino_probs)
]:
    acc, top5, ce = metrics(probs)
    print(f"{name}: Top-1={acc:.2f}% | Top-5={top5:.2f}% | CE={ce:.4f}")

# 7. Test ensemble weights
print("\n===== NEW ENSEMBLE =====")

results = []

for w in [0.0, 0.25, 0.50, 0.75, 1.0]:

    fused = w * conv_probs + (1 - w) * dino_probs

    acc, top5, ce = metrics(fused)
    results.append((acc, w, top5, ce))

    print(
        f"ConvNeXt {w:.0%} + DINOv2 {1-w:.0%}: "
        f"Top-1={acc:.2f}% | Top-5={top5:.2f}% | CE={ce:.4f}"
    )

best_acc, best_w, best_top5, best_ce = max(results)

print("\n===== BEST RESULT =====")
print(f"Best Top-1: {best_acc:.2f}%")
print(f"Best Top-5: {best_top5:.2f}%")
print(f"Cross-Entropy: {best_ce:.4f}")
print(f"ConvNeXt weight: {best_w:.2f}")
print(f"DINOv2 weight: {1-best_w:.2f}")

# 8. Save predictions and comparison
torch.save({
    "conv_probs": conv_probs,
    "dino_probs": dino_probs,
    "labels": labels,
    "classes": old["classes"],
    "best_weight": best_w,
    "best_accuracy": best_acc,
    "best_top5": best_top5,
    "best_cross_entropy": best_ce
}, f"{save_dir}/fusion_improved_validation.pth")

print("\nResults saved successfully!")


## Restore and unfreeze the last four blocks (source cell 98)

In [ ]:

import gc
import torch
import timm

from pathlib import Path
from torchvision import datasets, transforms as T
from torch.utils.data import DataLoader, Subset

# 1. Free memory occupied by experiments we cancelled
for name in ("d3_model", "d3_general", "dino_model", "fusion_cnn"):
    if name in globals():
        del globals()[name]

gc.collect()
torch.cuda.empty_cache()

device = torch.device("cuda")
save_dir = Path("/content/drive/MyDrive/NutriVision_AI")

# 2. Restore our best DINOv2 checkpoint
checkpoint = torch.load(
    save_dir / "dinov2_last2_best.pth",
    map_location="cpu",
    weights_only=True
)

d2_model = timm.create_model(
    "vit_base_patch14_dinov2.lvd142m",
    pretrained=False,
    num_classes=101,
    img_size=280
)

d2_model.load_state_dict(checkpoint["model_state_dict"])
d2_model = d2_model.to(device)

# 3. Freeze all parameters first
for p in d2_model.parameters():
    p.requires_grad = False

# 4. Unfreeze the last FOUR Transformer blocks
for block in d2_model.blocks[-4:]:
    for p in block.parameters():
        p.requires_grad = True

# Also train the normalization and classification layers
for module in (d2_model.norm, d2_model.head):
    for p in module.parameters():
        p.requires_grad = True

# 5. Recreate our DINOv2 image preprocessing
config = timm.data.resolve_model_data_config(d2_model)

mean = config["mean"]
std = config["std"]

train_transform = T.Compose([
    T.Resize(320, interpolation=T.InterpolationMode.BICUBIC),
    T.RandomCrop(280),
    T.RandomHorizontalFlip(),
    T.ColorJitter(
        brightness=0.1,
        contrast=0.1,
        saturation=0.1
    ),
    T.ToTensor(),
    T.Normalize(mean, std)
])

val_transform = T.Compose([
    T.Resize(320, interpolation=T.InterpolationMode.BICUBIC),
    T.CenterCrop(280),
    T.ToTensor(),
    T.Normalize(mean, std)
])

# 6. Load the SAME Food-101 split as our old experiments
train_source = datasets.Food101(
    root="/content/data",
    split="train",
    transform=train_transform,
    download=True
)

val_source = datasets.Food101(
    root="/content/data",
    split="train",
    transform=val_transform,
    download=False
)

split = torch.load(
    save_dir / "data_split.pth",
    map_location="cpu",
    weights_only=True
)

assert len(split["train_indices"]) == 68175
assert len(split["val_indices"]) == 7575

assert set(split["train_indices"]).isdisjoint(
    set(split["val_indices"])
)

if "classes" in checkpoint:
    assert list(checkpoint["classes"]) == list(train_source.classes)

train_dataset = Subset(
    train_source,
    split["train_indices"]
)

val_dataset = Subset(
    val_source,
    split["val_indices"]
)

train_loader = DataLoader(
    train_dataset,
    batch_size=8,
    shuffle=True,
    num_workers=0,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=16,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

trainable = sum(
    p.numel() for p in d2_model.parameters()
    if p.requires_grad
)

print("===== DINOv2 READY FOR IMPROVEMENT =====")
print("GPU:", torch.cuda.get_device_name(0))
print("Trainable parameters:", f"{trainable:,}")
print("Transformer blocks:", len(d2_model.blocks))
print("Unfrozen blocks:", 4)
print("Training images:", len(train_dataset))
print("Validation images:", len(val_dataset))
print("Image resolution:", 280)


## Train and save final released DINOv2 checkpoint (source cell 99)

In [ ]:

import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm

# 1. Different learning rates for different parts
optimizer = torch.optim.AdamW([
    {
        "params": list(d2_model.blocks[-4].parameters())
                  + list(d2_model.blocks[-3].parameters()),
        "lr": 3e-6,
        "weight_decay": 0.02
    },
    {
        "params": list(d2_model.blocks[-2].parameters())
                  + list(d2_model.blocks[-1].parameters()),
        "lr": 6e-6,
        "weight_decay": 0.02
    },
    {
        "params": d2_model.norm.parameters(),
        "lr": 1e-5,
        "weight_decay": 0.0
    },
    {
        "params": d2_model.head.parameters(),
        "lr": 3e-5,
        "weight_decay": 0.01
    }
])

train_criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
val_criterion = nn.CrossEntropyLoss()

epochs = 3
accumulation_steps = 2

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=epochs,
    eta_min=1e-7
)

assert torch.cuda.is_bf16_supported()

# Compare against our actual old ensemble
old_fusion = torch.load(
    save_dir / "fusion_improved_validation.pth",
    map_location="cpu",
    weights_only=True
)

best_ensemble = float(old_fusion["best_accuracy"])
best_single = 91.64
bad_epochs = 0

# 2. Training
for epoch in range(epochs):
    d2_model.train()
    optimizer.zero_grad(set_to_none=True)

    train_correct = 0
    train_total = 0

    for step, (images, labels) in enumerate(
        tqdm(train_loader, desc=f"Fine-tune {epoch+1}")
    ):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        with torch.autocast("cuda", dtype=torch.bfloat16):
            logits = d2_model(images)
            loss = train_criterion(logits.float(), labels)

        remaining = len(train_loader) - (
            step // accumulation_steps
        ) * accumulation_steps

        group_size = min(accumulation_steps, remaining)

        (loss / group_size).backward()

        if (
            (step + 1) % accumulation_steps == 0
            or step + 1 == len(train_loader)
        ):
            torch.nn.utils.clip_grad_norm_(
                d2_model.parameters(), 1.0
            )

            optimizer.step()
            optimizer.zero_grad(set_to_none=True)

        train_correct += (
            logits.argmax(1) == labels
        ).sum().item()

        train_total += labels.size(0)

    train_acc = 100 * train_correct / train_total

    # 3. Validation
    d2_model.eval()

    val_probs = []
    val_labels = []
    val_loss_sum = 0.0

    with torch.inference_mode():
        for images, labels in tqdm(
            val_loader,
            desc=f"Validate {epoch+1}"
        ):
            images = images.to(device)

            with torch.autocast("cuda", dtype=torch.bfloat16):
                logits = d2_model(images)

            loss = val_criterion(
                logits.float(),
                labels.to(device)
            )

            val_loss_sum += loss.item() * labels.size(0)

            val_probs.append(
                torch.softmax(logits.float(), dim=1).cpu()
            )

            val_labels.append(labels.cpu())

    probs = torch.cat(val_probs)
    labels = torch.cat(val_labels)

    # Check that old/new predictions are aligned
    assert torch.equal(labels, old_fusion["labels"])

    single_acc = (
        probs.argmax(1) == labels
    ).float().mean().item() * 100

    val_loss = val_loss_sum / len(labels)

    # 4. Compare using our ORIGINAL ensemble weights
    fused = (
        0.25 * old_fusion["conv_probs"]
        + 0.75 * probs
    )

    ensemble_acc = (
        fused.argmax(1) == labels
    ).float().mean().item() * 100

    ensemble_top5 = (
        fused.topk(5, dim=1).indices
        == labels[:, None]
    ).any(1).float().mean().item() * 100

    print(f"\n===== EPOCH {epoch+1}/{epochs} =====")
    print(f"Train Accuracy: {train_acc:.2f}%")
    print(f"DINOv2 Validation: {single_acc:.2f}%")
    print(f"Ensemble Validation: {ensemble_acc:.2f}%")
    print(f"Ensemble Top-5: {ensemble_top5:.2f}%")
    print(f"Validation Loss: {val_loss:.4f}")

    # 5. Save if the overall system improves
    if ensemble_acc > best_ensemble + 0.001:
        best_ensemble = ensemble_acc
        bad_epochs = 0

        torch.save({
            "model_state_dict": d2_model.state_dict(),
            "epoch": epoch + 1,
            "single_accuracy": single_acc,
            "ensemble_accuracy": ensemble_acc,
            "class_names": train_source.classes
        }, save_dir / "dinov2_last4_best.pth")

        torch.save({
            "dino_probs": probs,
            "conv_probs": old_fusion["conv_probs"],
            "labels": labels,
            "ensemble_accuracy": ensemble_acc
        }, save_dir / "dinov2_last4_fusion_best.pth")

        print("NEW BEST ENSEMBLE SAVED!")

    else:
        bad_epochs += 1
        print("No ensemble improvement.")

    scheduler.step()

    print(f"Best Ensemble: {best_ensemble:.2f}%")

    if bad_epochs >= 2:
        print("Early stopping.")
        break

print("\nExperiment completed.")
print(f"Best Ensemble Accuracy: {best_ensemble:.2f}%")
